This is a markdown cell. It describes the purpose of this notebook and its code.The other cells are coding cells. 

This notebook will extract the information from the scanned Manorial Records and populate excel files with them. 

In [2]:
!pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 20.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 19.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.5/502.5 kB 10.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 10.7 MB/s eta 0:00:00


In [4]:
!pip install PyPDF2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.7 MB/s eta 0:00:00a 0:00:01


In [15]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 17.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.0/49.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 17.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 17.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 19.7 MB/s eta 0:00:0000:0100:01


In [35]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.6/239.6 kB 5.2 MB/s eta 0:00:00:00:01


In [20]:
!pip install openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 4.4 MB/s eta 0:00:0000:01


In [1]:
# Import the pandas library with an alias 'pd' (this is a common convention)
import pandas as pd

# Import libraries for pdf reading
from PyPDF2 import PdfReader

# Better use this as it recognizes fonts: 
import pdfplumber

# Import openpyxl to use its functionality for working with Excel files
import openpyxl

# Extracting from word document 
import docx

#Import regular expressionsio
import re

In [63]:
####################### SCRIPT 7 #######################

import os
import glob
from pathlib import Path
from copy import copy as pycopy
from openpyxl import load_workbook
from openpyxl.cell.cell import MergedCell

# ---------- PATHS ----------
input_base_path = r"C:\Users\kubak\Documents\GitHub\student_assistant_manorial_records\input\OCR 1301_unfinished"
output_base_path = r"C:\Users\kubak\Documents\GitHub\student_assistant_manorial_records\output\OCR 1301-1302_unifinished\script7"
template_base_path = r"C:\Users\kubak\Documents\GitHub\student_assistant_manorial_records\archive"
template_filename = "Adderbury 1409_wl_format.xlsx"

Path(output_base_path).mkdir(parents=True, exist_ok=True)
template_path = os.path.join(template_base_path, template_filename)

# ---------- FUNCTIONS & VARIABLES ----------
TRAIL_PUNCT = (":", ";", ".", ",")

OVERVIEW_TO_RECEIPTS_MAP = {"rents of assize": "rents"}
OVERVIEW_TO_EXPENSES_MAP = {
    "steward's expenses": "expenses of the steward",
}

GRAIN_MAP = {
    "wheat": "wheat",
    "rye": None,
    "dredge": None,
    "curall": "curall",
    "barley": "barley",
    "oats": "oats",
    "beans": None,
    "peas": "peas",
    "vetches": "vetches",
}

def normalize_label(s):
    if s is None:
        return ""
    if not isinstance(s, str):
        s = str(s)
    s = s.replace("\u00A0", " ")
    s = " ".join(s.split()).strip().lower()
    while s.endswith(TRAIL_PUNCT):
        s = s[:-1].strip()
    return s

def strip_total(s):
    s = normalize_label(s)
    if s.endswith(" totals"):
        return s[:-7].strip()
    if s.endswith(" total"):
        return s[:-6].strip()
    return s

def overview_to_key(overview_label, mapping_dict):
    s = strip_total(overview_label)
    for phrase, repl in mapping_dict.items():
        if s.startswith(phrase):
            return repl
    return s

def build_last_row_index_multi(ws, start_row=3, label_col=1):
    last = {}
    for r in range(start_row, ws.max_row + 1):
        v = ws.cell(row=r, column=label_col).value
        key1 = normalize_label(v)
        if not key1:
            continue
        key2 = strip_total(key1)
        last[key1] = r
        last[key2] = r
    return last

def find_last_row_startswith(ws, key, start_row=3, label_col=1):
    key = normalize_label(key)
    if not key:
        return None
    last = None
    for r in range(start_row, ws.max_row + 1):
        v = normalize_label(ws.cell(row=r, column=label_col).value)
        if v.startswith(key):
            last = r
    return last

def find_sheet_by_keyword(wb, keyword):
    if keyword in wb.sheetnames:
        return keyword
    kw = keyword.lower()
    for name in wb.sheetnames:
        if kw in name.lower():
            return name
    return None

def find_grange_sheet(wb):
    for nm in wb.sheetnames:
        ln = nm.lower()
        if "issues" in ln and "grange" in ln:
            return nm
    return None

def find_stock_sheet(wb):
    for nm in wb.sheetnames:
        if nm.strip().lower() == "stock":
            return nm
    for nm in wb.sheetnames:
        if "stock" in nm.lower():
            return nm
    return None

def to_number(x):
    if x is None:
        return None
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        s = x.strip().replace("\u00A0", " ").replace(" ", "")
        if "," in s and "." not in s:
            s = s.replace(",", ".")
        try:
            return float(s)
        except ValueError:
            return None
    return None

def num_or_zero(x):
    n = to_number(x)
    return 0.0 if n is None else n

def clean_excel_formula(v):
    if isinstance(v, str) and v.startswith("=+"):
        return "=" + v[2:]
    return v

# ---------- COPY ROUTINE (VALUES + FORMATTING) ----------
def copy_sheet_full(src_ws, dest_ws):
    dest_ws.freeze_panes = src_ws.freeze_panes
    if src_ws.auto_filter and src_ws.auto_filter.ref:
        dest_ws.auto_filter.ref = src_ws.auto_filter.ref

    for col_letter, dim in src_ws.column_dimensions.items():
        d = dest_ws.column_dimensions[col_letter]
        d.width = dim.width
        d.hidden = dim.hidden
        d.outlineLevel = dim.outlineLevel
        d.collapsed = dim.collapsed

    for row_idx, dim in src_ws.row_dimensions.items():
        d = dest_ws.row_dimensions[row_idx]
        d.height = dim.height
        d.hidden = dim.hidden
        d.outlineLevel = dim.outlineLevel
        d.collapsed = dim.collapsed

    for merged_range in src_ws.merged_cells.ranges:
        dest_ws.merge_cells(str(merged_range))

    for row in src_ws.iter_rows():
        for c in row:
            if isinstance(c, MergedCell):
                continue
            dc = dest_ws.cell(row=c.row, column=c.column, value=c.value)

            if c.has_style:
                dc._style = pycopy(c._style)
                dc.font = pycopy(c.font)
                dc.border = pycopy(c.border)
                dc.fill = pycopy(c.fill)
                dc.number_format = c.number_format
                dc.protection = pycopy(c.protection)
                dc.alignment = pycopy(c.alignment)

            if c.hyperlink:
                dc._hyperlink = pycopy(c.hyperlink)
            if c.comment:
                dc.comment = pycopy(c.comment)

    dest_ws.page_margins = pycopy(src_ws.page_margins)
    dest_ws.page_setup = pycopy(src_ws.page_setup)
    dest_ws.print_options = pycopy(src_ws.print_options)

# ---------- PROCESS ALL _wl FILES ----------
pattern = os.path.join(input_base_path, "*_wl.xlsx")
input_files = sorted(glob.glob(pattern))
print(f"Found {len(input_files)} input workbooks")

for input_file in input_files:
    src_filename = os.path.basename(input_file)
    base_no_ext = os.path.splitext(src_filename)[0]
    manor_name = base_no_ext.split("_")[0]

    output_filename = f"{base_no_ext}_7.xlsx"
    output_path = os.path.join(output_base_path, output_filename)

    # 1) template -> output base
    wb_out = load_workbook(template_path, data_only=False)
    if "Adderbury Overview" not in wb_out.sheetnames:
        wb_out.close()
        raise RuntimeError(f"Template error: 'Adderbury Overview' sheet not found in {template_filename}")

    ws_overview = wb_out["Adderbury Overview"]
    ws_overview.title = f"{manor_name} Overview - Completed"
    ws_overview["A1"].value = f"{manor_name.upper()} OVERVIEW - COMPLETED"

    for sh in list(wb_out.sheetnames):
        if sh.startswith("Adderbury ") and sh != ws_overview.title:
            wb_out.remove(wb_out[sh])

    # 2) copy input workbook sheets (format preserved) into output
    wb_in = load_workbook(input_file, data_only=False)
    wb_in_vals = load_workbook(input_file, data_only=True)

    for src_ws in wb_in.worksheets:
        src_title = src_ws.title
        if src_title in wb_out.sheetnames:
            wb_out.remove(wb_out[src_title])
        dest_ws = wb_out.create_sheet(src_title)
        copy_sheet_full(src_ws, dest_ws)

    # ---------- RECEIPTS (B4:B22) ----------
    receipts_out_name = find_sheet_by_keyword(wb_out, "Receipts") or find_sheet_by_keyword(wb_out, "receipt")
    receipts_in_name  = find_sheet_by_keyword(wb_in_vals, "Receipts") or find_sheet_by_keyword(wb_in_vals, "receipt")
    if receipts_out_name and receipts_in_name:
        ws_receipts_out = wb_out[receipts_out_name]
        ws_receipts_val = wb_in_vals[receipts_in_name]
        last_rows = build_last_row_index_multi(ws_receipts_out, start_row=3, label_col=1)

        for r in range(4, 23):
            key = overview_to_key(ws_overview.cell(row=r, column=1).value, OVERVIEW_TO_RECEIPTS_MAP)
            if not key:
                continue

            match_row = last_rows.get(key)
            if match_row is None and key.endswith("s"):
                match_row = last_rows.get(key[:-1])
            if match_row is None:
                match_row = find_last_row_startswith(ws_receipts_out, key, start_row=3, label_col=1)

            if match_row is None:
                ws_overview.cell(row=r, column=2).value = None
                continue

            cached_val = ws_receipts_val.cell(row=match_row, column=6).value
            ws_overview.cell(row=r, column=2).value = cached_val if cached_val is not None else clean_excel_formula(ws_receipts_out.cell(row=match_row, column=6).value)

    # ---------- EXPENSES (B30:B43) ----------
    expenses_out_name = find_sheet_by_keyword(wb_out, "Expenses") or find_sheet_by_keyword(wb_out, "expense")
    expenses_in_name  = find_sheet_by_keyword(wb_in_vals, "Expenses") or find_sheet_by_keyword(wb_in_vals, "expense")
    if expenses_out_name and expenses_in_name:
        ws_expenses_out = wb_out[expenses_out_name]
        ws_expenses_val = wb_in_vals[expenses_in_name]
        last_rows_exp = build_last_row_index_multi(ws_expenses_out, start_row=3, label_col=1)

        for r in range(30, 44):
            key = overview_to_key(ws_overview.cell(row=r, column=1).value, OVERVIEW_TO_EXPENSES_MAP)
            if not key:
                continue

            match_row = last_rows_exp.get(key)
            if match_row is None and key.endswith("s"):
                match_row = last_rows_exp.get(key[:-1])
            if match_row is None:
                match_row = find_last_row_startswith(ws_expenses_out, key, start_row=3, label_col=1)

            if match_row is None:
                ws_overview.cell(row=r, column=2).value = None
                continue

            cached_val = ws_expenses_val.cell(row=match_row, column=6).value
            ws_overview.cell(row=r, column=2).value = cached_val if cached_val is not None else clean_excel_formula(ws_expenses_out.cell(row=match_row, column=6).value)

    # ---------- ISSUES OF THE GRANGE ----------
    grange_in_name = find_grange_sheet(wb_in_vals)
    if grange_in_name:
        ws_grange_val = wb_in_vals[grange_in_name]

        lookup_1_9 = {}
        for rr in range(1, 10):
            k = normalize_label(ws_grange_val.cell(row=rr, column=1).value)
            if k:
                lookup_1_9[k] = ws_grange_val.cell(row=rr, column=2).value

        for ov_row in range(62, 71):
            ov_key = normalize_label(ws_overview.cell(row=ov_row, column=1).value)
            ws_overview.cell(row=ov_row, column=2).value = lookup_1_9.get(ov_key, None)

        grange_o_lookup = {}
        for rr in range(11, ws_grange_val.max_row + 1):
            k = normalize_label(ws_grange_val.cell(row=rr, column=1).value)
            if k and k not in grange_o_lookup:
                grange_o_lookup[k] = ws_grange_val.cell(row=rr, column=15).value  # O

        for ov_row in range(61, 71):
            ov_label = normalize_label(ws_overview.cell(row=ov_row, column=1).value)
            mapped = GRAIN_MAP.get(ov_label, None)
            ws_overview.cell(row=ov_row, column=5).value = grange_o_lookup.get(mapped, None) if mapped else None

    # ---------- STOCK + WOOL + POULTRY ----------
    stock_in_name = find_stock_sheet(wb_in_vals)
    if stock_in_name:
        ws_stock_val = wb_in_vals[stock_in_name]

        # STOCK rows 5..30 -> overview rows 87..112
        stock_row_for_key, stock_B, stock_K, stock_O = {}, {}, {}, {}
        for rr in range(5, 31):
            k = normalize_label(ws_stock_val.cell(row=rr, column=1).value)
            if not k:
                continue
            stock_row_for_key[k] = rr
            stock_B[k] = ws_stock_val.cell(row=rr, column=2).value   # B
            stock_K[k] = ws_stock_val.cell(row=rr, column=11).value  # K
            stock_O[k] = ws_stock_val.cell(row=rr, column=15).value  # O

        for ov_row in range(87, 113):
            k = normalize_label(ws_overview.cell(row=ov_row, column=1).value)
            if not k:
                continue

            b = stock_B.get(k, None)
            c = stock_K.get(k, None)
            o = stock_O.get(k, None)
            rr = stock_row_for_key.get(k, ov_row - 82)

            ws_overview.cell(row=ov_row, column=2).value = b  # B
            ws_overview.cell(row=ov_row, column=3).value = c  # C

            nb, nc, no = to_number(b), to_number(c), to_number(o)

            if nb is not None and nc is not None:
                d_val = nc - nb
                ws_overview.cell(row=ov_row, column=4).value = d_val
            else:
                d_val = None
                ws_overview.cell(row=ov_row, column=4).value = f"=C{ov_row}-B{ov_row}"

            if nc is not None and no is not None:
                ws_overview.cell(row=ov_row, column=6).value = nc * no
            else:
                ws_overview.cell(row=ov_row, column=6).value = f"='{stock_in_name}'!K{rr}*'{stock_in_name}'!O{rr}"

            if d_val is not None and no is not None:
                ws_overview.cell(row=ov_row, column=7).value = d_val * no
            else:
                ws_overview.cell(row=ov_row, column=7).value = f"=D{ov_row}*'{stock_in_name}'!O{rr}"

        # WOOL rows 35..38 -> overview rows 118..122 (restored exactly)
        wool_row_for_key, wool_B, wool_K, wool_C, wool_O = {}, {}, {}, {}, {}
        for rr in range(35, 39):
            k = normalize_label(ws_stock_val.cell(row=rr, column=1).value)
            if not k:
                continue
            wool_row_for_key[k] = rr
            wool_B[k] = ws_stock_val.cell(row=rr, column=2).value    # B -> Ov B
            wool_K[k] = ws_stock_val.cell(row=rr, column=11).value   # K -> Ov C
            wool_C[k] = ws_stock_val.cell(row=rr, column=3).value    # C -> Ov D
            wool_O[k] = ws_stock_val.cell(row=rr, column=15).value   # O

        for ov_row in range(118, 123):
            k = normalize_label(ws_overview.cell(row=ov_row, column=1).value)
            if not k:
                continue
            rr = wool_row_for_key.get(k, None)

            ws_overview.cell(row=ov_row, column=2).value = wool_B.get(k, None)
            ws_overview.cell(row=ov_row, column=3).value = wool_K.get(k, None)
            ws_overview.cell(row=ov_row, column=4).value = wool_C.get(k, None)

            nc = to_number(wool_K.get(k, None))   # Overview C
            no = to_number(wool_O.get(k, None))
            if nc is not None and no is not None:
                ws_overview.cell(row=ov_row, column=6).value = nc * no
            else:
                ws_overview.cell(row=ov_row, column=6).value = f"=C{ov_row}*'{stock_in_name}'!O{rr}" if rr else None

        # Helper: generic overview block from stock rows (B,K,O; D=B-C; F=C*O) with default 0
        def fill_overview_block_from_stock(ov_start, ov_end, stock_row_start, stock_row_end):
            row_for_key, val_B, val_K, val_O = {}, {}, {}, {}
            for rr in range(stock_row_start, stock_row_end + 1):
                k = normalize_label(ws_stock_val.cell(row=rr, column=1).value)
                if not k:
                    continue
                row_for_key[k] = rr
                val_B[k] = ws_stock_val.cell(row=rr, column=2).value
                val_K[k] = ws_stock_val.cell(row=rr, column=11).value
                val_O[k] = ws_stock_val.cell(row=rr, column=15).value

            for ov_row in range(ov_start, ov_end + 1):
                k = normalize_label(ws_overview.cell(row=ov_row, column=1).value)
                if not k:
                    continue
                rr = row_for_key.get(k, None)

                b = num_or_zero(val_B.get(k, None))
                c = num_or_zero(val_K.get(k, None))
                ws_overview.cell(row=ov_row, column=2).value = b
                ws_overview.cell(row=ov_row, column=3).value = c
                ws_overview.cell(row=ov_row, column=4).value = b - c  # D = B - C

                no = to_number(val_O.get(k, None))
                if no is not None:
                    ws_overview.cell(row=ov_row, column=6).value = c * no
                else:
                    ws_overview.cell(row=ov_row, column=6).value = f"=C{ov_row}*'{stock_in_name}'!O{rr}" if rr else 0

        # Poultry blocks
        fill_overview_block_from_stock(151, 154, 68, 71)   # Poultry Stock
        fill_overview_block_from_stock(159, 160, 76, 77)   # Poultry Production
        fill_overview_block_from_stock(166, 168, 88, 93)   # Next poultry section

    # ---------------- SUMMARY ROWS (Overview - Completed) ----------------
    # ws_overview already refers to the renamed template sheet:
    # ws_overview.title = f"{manor_name} Overview - Completed"

    # Row 23: sum B5:B22
    ws_overview["B23"].value = "=SUM(B5:B22)"

    # Row 24: sum B4:B22
    ws_overview["B24"].value = "=SUM(B4:B22)"

    # Row 25: pull from Receipts -> row where col A == "Total of all receipts recorded in accounts", value in col F
    target_label = "total of all receipts recorded in accounts"

    receipts_out_name = find_sheet_by_keyword(wb_out, "Receipts") or find_sheet_by_keyword(wb_out, "receipt")
    receipts_in_name  = find_sheet_by_keyword(wb_in_vals, "Receipts") or find_sheet_by_keyword(wb_in_vals, "receipt")

    found_row = None
    if receipts_out_name and receipts_in_name:
        receipts_ws_out = wb_out[receipts_out_name]
        receipts_ws_val = wb_in_vals[receipts_in_name]

        for r in range(1, receipts_ws_out.max_row + 1):
            a_val = receipts_ws_out.cell(row=r, column=1).value  # Column A label
            if a_val is None:
                continue
            if normalize_label(a_val) == target_label:
                found_row = r
                break

        if found_row:
            # Use cached numeric value if the cell is a formula in Receipts
            cached_val = receipts_ws_val.cell(row=found_row, column=6).value  # Column F
            ws_overview["B25"].value = cached_val if cached_val is not None else clean_excel_formula(
                receipts_ws_out.cell(row=found_row, column=6).value
            )
        else:
            ws_overview["B25"].value = None
    else:
        ws_overview["B25"].value = None

    # Row 26: B25 + B4 (Total recorded incl. arrears)
    ws_overview["B26"].value = "=B25+B4"

    # ---------------- SUMMARY ROWS: EXPENSES TOTALS (Overview - Completed) ----------------
    # Fill Overview B44:B47 from Expenses (match labels in col A, take value from col F)

    expenses_out_name = find_sheet_by_keyword(wb_out, "Expenses") or find_sheet_by_keyword(wb_out, "expense")
    expenses_in_name  = find_sheet_by_keyword(wb_in_vals, "Expenses") or find_sheet_by_keyword(wb_in_vals, "expense")

    expense_targets = {
        44: "total of all expenses per account",
        45: "total of all expenses (calculated here)",
        46: "total fixed investment",
        47: "total demesne fixed investment",
    }

    if expenses_out_name and expenses_in_name:
        ws_exp_out = wb_out[expenses_out_name]
        ws_exp_val = wb_in_vals[expenses_in_name]

        # Build a quick lookup from normalized label -> last row where it occurs
        exp_last = {}
        for r in range(1, ws_exp_out.max_row + 1):
            key = normalize_label(ws_exp_out.cell(row=r, column=1).value)
            if key:
                exp_last[key] = r

        for ov_row, lbl in expense_targets.items():
            match_row = exp_last.get(lbl)
            if match_row:
                cached_val = ws_exp_val.cell(row=match_row, column=6).value  # F
                ws_overview.cell(row=ov_row, column=2).value = (
                    cached_val if cached_val is not None
                    else clean_excel_formula(ws_exp_out.cell(row=match_row, column=6).value)
                )
            else:
                ws_overview.cell(row=ov_row, column=2).value = None
    else:
        # Expenses sheet missing; leave blanks
        for ov_row in expense_targets.keys():
            ws_overview.cell(row=ov_row, column=2).value = None

    # ---------------- OTHER SUMMARY ROWS (Overview - Completed) ----------------
    # Fill Overview B50, B52, B53 from Expenses (match labels in col A, take value from col F)

    expenses_out_name = find_sheet_by_keyword(wb_out, "Expenses") or find_sheet_by_keyword(wb_out, "expense")
    expenses_in_name  = find_sheet_by_keyword(wb_in_vals, "Expenses") or find_sheet_by_keyword(wb_in_vals, "expense")

    if expenses_out_name and expenses_in_name:
        ws_exp_out = wb_out[expenses_out_name]
        ws_exp_val = wb_in_vals[expenses_in_name]

        # Helper: find FIRST row where column A matches a target label (normalized)
        def find_first_row_exact(ws, target_norm, col=1):
            for r in range(1, ws.max_row + 1):
                if normalize_label(ws.cell(row=r, column=col).value) == target_norm:
                    return r
            return None

        # Row 50: FIRST instance of "Total of all expenses" (second instance is incorrect)
        r50 = find_first_row_exact(ws_exp_out, "total of all expenses", col=1)
        if r50:
            cached_val = ws_exp_val.cell(row=r50, column=6).value  # F
            ws_overview["B50"].value = cached_val if cached_val is not None else clean_excel_formula(
                ws_exp_out.cell(row=r50, column=6).value
            )
        else:
            ws_overview["B50"].value = None

        # Row 52: "Allowances without writ total"
        r52 = find_first_row_exact(ws_exp_out, "allowances without writ total", col=1)
        if r52:
            cached_val = ws_exp_val.cell(row=r52, column=6).value  # F
            ws_overview["B52"].value = cached_val if cached_val is not None else clean_excel_formula(
                ws_exp_out.cell(row=r52, column=6).value
            )
        else:
            ws_overview["B52"].value = None

        # Row 53: "Cash deliveries"
        r53 = find_first_row_exact(ws_exp_out, "cash deliveries", col=1)
        if r53:
            cached_val = ws_exp_val.cell(row=r53, column=6).value  # F
            ws_overview["B53"].value = cached_val if cached_val is not None else clean_excel_formula(
                ws_exp_out.cell(row=r53, column=6).value
            )
        else:
            ws_overview["B53"].value = None

    else:
        ws_overview["B50"].value = None
        ws_overview["B52"].value = None
        ws_overview["B53"].value = None


    # ---------------- ADDITIONAL SUMMARY CELLS (Overview - Completed) ----------------

    # G113: sum G87:G112
    ws_overview["G113"].value = "=SUM(G87:G112)"

    # F114: sum Stock!T5:T30
    stock_out_name = find_stock_sheet(wb_out)
    stock_in_name  = find_stock_sheet(wb_in_vals)

    if stock_out_name and stock_in_name:
        # Use a formula reference so it stays transparent in Excel
        ws_overview["F114"].value = f"=SUM('{stock_out_name}'!T5:T30)"
    else:
        ws_overview["F114"].value = None

    # F115: sum Stock!R5:R30
    stock_out_name = find_stock_sheet(wb_out)
    stock_in_name  = find_stock_sheet(wb_in_vals)

    if stock_out_name and stock_in_name:
        ws_overview["F115"].value = f"=SUM('{stock_out_name}'!R5:R30)"
    else:
        ws_overview["F115"].value = None

    # F156: sum Stock!R67:R70
    stock_out_name = find_stock_sheet(wb_out)
    stock_in_name  = find_stock_sheet(wb_in_vals)

    if stock_out_name and stock_in_name:
        ws_overview["F156"].value = f"=SUM('{stock_out_name}'!R67:R70)"
    else:
        ws_overview["F156"].value = None

    # ---------------- END CHANGES ----------------
    wb_in.close()
    wb_in_vals.close()


# ---------- UNFREEZE PANES + NORMALIZE SHEET VIEWS ----------
    for ws in wb_out.worksheets:
        ws.freeze_panes = None
        sv = ws.sheet_view
        sv.workbookViewId = 0
        sv.topLeftCell = "A1"
        sv.zoomScale = None
        sv.zoomScaleNormal = None
        sv.zoomScalePageLayoutView = None
        sv.zoomScaleSheetLayoutView = None
        sv.view = None
        sv.rightToLeft = None
        sv.tabSelected = None
        sv.selection = [Selection(activeCell="A1", sqref="A1")]

    # ---------- WORKBOOK-LEVEL SANITATION ----------
    wb_out.calculation.fullCalcOnLoad = True
    wb_out.active = 0
    try:
        for dn in list(wb_out.defined_names.definedName):
            try:
                del wb_out.defined_names[dn.name]
            except Exception:
                pass
    except Exception:
        pass

    # ---------- SAVE ----------
    wb_out.save(output_path)
    wb_out.close()

    # ---------- PATCH RELS TARGETS (optional but enabled) ----------

    print(f"Created: {output_filename}")



Found 40 input workbooks
Created: Alverstoke_1301_wl_7.xlsx
Created: Bentley_1301_wl_7.xlsx
Created: BishopsFonthill 1301_wl_7.xlsx
Created: BishopsSutton_1301_wl_7.xlsx
Created: BishopsWaltham_1301_wl_7.xlsx
Created: Bishopstone_1301_wl_7.xlsx
Created: Bitterne_1301_wl_7.xlsx
Created: Brightwell_1301_wl_7.xlsx
Created: Burghclere_1301_wl_7.xlsx
Created: Cams_1301_wl_7.xlsx
Created: Cheriton_1301_wl_7.xlsx
Created: Crawley 1301_wl_7.xlsx
Created: Culham_1301_wl_7.xlsx
Created: Droxford_1301_wl_7.xlsx
Created: EastMeonChurch_1301_wl_7.xlsx
Created: EastMeon_1301_wl_7.xlsx
Created: Ecchinswell_1301_wl_7.xlsx
Created: Esher_1301_wl_7.xlsx
Created: Farnham_1301_wl_7.xlsx
Created: Gosport_1301_wl_7.xlsx
Created: Hambledon_1301_wl_7.xlsx
Created: Harwell_1301_wl_7.xlsx
Created: Havant_1301_wl_7.xlsx
Created: Highclere_1301_wl_7.xlsx
Created: HindonBorough_1301_wl_7.xlsx
Created: Ivinghoe_1301_wl_7.xlsx
Created: Marshalsea_1301_wl_7.xlsx
Created: Merdon 1301_wl_7.xlsx
Created: Morton_1301_wl_

In [ ]:
####################### OPTIONAL #######################

# This cell removes the incomplete overview sheet from all cells. Run only for convenience. 